In [57]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import random

%matplotlib inline

In [2]:
#dataset



X = np.load("data/small/X.npy", mmap_mode="r")
Y = np.load("data/small/policy.npy")

X = torch.from_numpy(X)
Y = torch.from_numpy(Y).long()


C:\Users\Faure\AppData\Local\Temp\ipykernel_17108\3009614510.py:8: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  X = torch.from_numpy(X)


In [3]:
N = len(X)
n1 = int(0.8*N)
n2=int(0.9*N)


g = torch.Generator().manual_seed(44)
perm = torch.randperm(N, generator=g)
tr, dev, te = perm[:n1], perm[n1:n2], perm[n2:]

Xtr, Ytr = X[tr], Y[tr]
Xdev, Ydev  = X[dev], Y[dev]
Xte, Yte = X[te], Y[te]
del X, Y

In [4]:
Xtr.shape, Ytr.shape

(torch.Size([7055, 18, 8, 8]), torch.Size([7055]))

In [ ]:
class Linear():

    def __init__(self, fan_in, fan_out, bias = False, unf=True):
        self.weight = torch.randn((fan_out, fan_in))/(fan_in**0.5)
        self.bias = torch.zeros((fan_out,1)) if bias else None
        self.unf = unf

    def __call__(self, x):
        B = x.shape[0]
        if self.unf:
            x = unfoldX(x)
        else:
            x = x.view(B, x.shape[1], -1) #(B,C,64)
        self.out =  self.weight @ x
        if self.bias is not None:
            self.out += self.bias
        return self.out.view(B, -1, 8, 8)

class LinearFlat():
    def __init__(self, fan_in, fan_out, bias=True):
        
        self.weight = torch.randn((fan_out, fan_in)) / (fan_in ** 0.5)
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        out = x @ self.weight.T
        if self.bias is not None:
            out = out + self.bias
        return out
    
class BatchNorm2d():
    def __init__(self, dim, eps=1e-5, momentum=0.1, device=None):
        self.eps = eps
        self.momentum = momentum
        self.training = True

        self.gamma = torch.ones(dim, device=device, requires_grad=True)
        self.beta = orch.zeros(dim, device=device, requires_grad=True)

        running_mean = torch.zeros(dim) 
        running_var = torch.ones(dim)

    def forward(self, x):
        if self.training:
            mean = x.mean(dim=(0, 2, 3))
            var = x.var(dim=(0, 2, 3), unbiased=False)
            with torch.no_grad():
                self.running_mean.mul_(1 - self.momentum).add_(self.momentum * mean)
                self.running_var.mul_(1 - self.momentum).add_(self.momentum * var)
        else:
            mean, var = self.running_mean, self.running_var

        shape = (1, -1, 1, 1)
        xhat = (x - mean.view(shape)) / torch.sqrt(var.view(shape) + self.eps)
        return self.gamma.view(shape) * xhat + self.beta.view(shape)


class Tanh():

    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []

class reLU():

    def __call__(self, x):
        self.out = x * (x>0)
        return self.out

    def parameters(self):
        return []


In [47]:
n_hidden = 128
n_trans = 162 #18*3*3
n_final = 73
unfold = 3



In [ ]:
Entrance = [Linear(n_trans, n_hidden), BatchNorm2d(n_hidden), reLU()]

repeatedBlock1 = [Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden), reLU(), Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden)]
repeatedBlock2 = [Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden), reLU(), Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden)]
repeatedBlock3 = [Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden), reLU(), Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden)]
repeatedBlock4 = [Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden), reLU(), Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden)]
repeatedBlock5 = [Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden), reLU(), Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden)]
repeatedBlock6 = [Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden), reLU(), Linear(n_hidden*unfold**2, n_hidden), BatchNorm2d(n_hidden)]

policyOutput = [Linear(n_hidden, n_final, unf = False)]
valueOutput = [
    Linear(n_hidden, 1, unf=False),
    BatchNorm2d(1),
    reLU(),
]
valueLinear1 = LinearFlat(256, 64, bias=True)
valueLinear2 = LinearFlat(1, 256, bias=True)

with torch.no_grad():
    policyOutput[0].weight *= 0.1 # last layer less confident
    for l in Entrance:
        if isinstance(l, Linear):
            l.weight *= 2**0.5
    for l in repeatedBlock1:
            if isinstance(l, Linear):
                l.weight *=  2**0.5
    for l in repeatedBlock2:
                if isinstance(l, Linear):
                    l.weight *=  2**0.5
    for l in repeatedBlock3:
                if isinstance(l, Linear):
                    l.weight *=  2**0.5
    for l in repeatedBlock4:
                if isinstance(l, Linear):
                    l.weight *=  2**0.5
    for l in repeatedBlock5:
                if isinstance(l, Linear):
                    l.weight *=  2**0.5
    for l in repeatedBlock6:
                if isinstance(l, Linear):
                    l.weight *=  2**0.5

parameters = [p for layer in valueOutput for p in layer.parameters()] + [p for p in valueLinear1.parameters()] + [p for p in valueLinear2.parameters()] + [p for layer in Entrance for p in layer.parameters()] + [p for layer in repeatedBlock1 for p in layer.parameters()] + [p for layer in repeatedBlock2 for p in layer.parameters()]  +[p for layer in repeatedBlock3 for p in layer.parameters()]  +[p for layer in repeatedBlock4 for p in layer.parameters()]  +[p for layer in repeatedBlock5 for p in layer.parameters()]  +[p for layer in repeatedBlock6 for p in layer.parameters()]  +  [p for layer in policyOutput for p in layer.parameters()] 

print(sum(p.nelement() for p in parameters))

for p in parameters:
    p.requires_grad = True




1819907


In [49]:
def unfoldX(x):
    B, C, H, W = x.shape
    xp = F.pad(x, ((unfold-1)//2, (unfold-1)//2, (unfold-1)//2, (unfold-1)//2))          
    slices = [xp[:, :, u:u+H, v:v+W] for u in range(unfold) for v in range(unfold)]
    return torch.stack(slices, dim=2).reshape(B, C * unfold*unfold, H * W)

In [50]:
#forward pass
relu = reLU()
def forward(x):
    
    for layer in Entrance:
        x = layer(x)

    x0 = x
    for layer in repeatedBlock1:
        x = layer(x)
    x= x+ x0
    x=relu(x)
    x1 = x
    for layer in repeatedBlock2:
            x = layer(x)
    x= x+ x1
    x=relu(x)

    x2=x
    for layer in repeatedBlock3:
            x = layer(x)
    x= x+ x2
    x=relu(x)

    x3=x
    for layer in repeatedBlock4:
            x = layer(x)
    x=x3 +x
    x=relu(x)

    x4=x
    for layer in repeatedBlock5:
            x = layer(x)
    x=x4+x
    x=relu(x)

    x5=x
    for layer in repeatedBlock6:
            x = layer(x)
    x=x5 +x
    x=relu(x)

    v = x
    for layer in valueOutput:
        v = layer(v)
    v = v.view(v.shape[0], -1)      # (B, 64)
    v = relu(valueLinear1(v))
    v = torch.tanh(valueLinear2(v))

    for layer in policyOutput:
        x = layer(x)

    x = x.permute(0, 2, 3, 1).reshape(x.shape[0], -1)
    
    return x, v

    
    

In [51]:
B = 1
for case, type_ in [(0, 0), (27, 40), (63, 72)]:
    x = torch.zeros(B, 73, 8, 8)
    r, f = case // 8, case % 8
    x[0, type_, r, f] = 1.0
    out = x.permute(0, 2, 3, 1).reshape(B, -1)
    attendu = 73 * case + type_
    print(case, type_, out.argmax().item(), attendu, out.argmax().item() == attendu)

0 0 0 0 True
27 40 2011 2011 True
63 72 4671 4671 True


In [56]:
xb = Xtr[:8].float()
logits, v = forward(xb)
print(logits.shape, v.shape)
print(F.cross_entropy(logits, Ytr[:8]).item())

torch.Size([8, 4672]) torch.Size([8, 1])
8.689288139343262


In [93]:
#doing it better and shorter:
class Linear(nn.Module):

    def __init__(self, fan_in, fan_out, bias = False, unf=True):
        super().__init__()  
        self.weight = nn.Parameter(torch.randn((fan_out, fan_in))/(fan_in**0.5))
        self.bias = nn.Parameter(torch.zeros((fan_out,1))) if bias else None
        self.unf = unf

    def forward(self, x):
        B = x.shape[0]
        if self.unf:
            C = self.weight.shape[1] // (unfold*unfold)
            w = self.weight.view(-1, C, unfold, unfold)
            return F.conv2d(x, w, padding=1)
        x = x.view(B, x.shape[1], -1)
        out = self.weight @ x
        return out.view(B, -1, 8, 8)

class LinearFlat(nn.Module):
    def __init__(self, fan_in, fan_out, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.randn((fan_out, fan_in)) / (fan_in ** 0.5))
        self.bias = nn.Parameter(torch.zeros(fan_out)) if bias else None

    def forward(self, x):
        out = x @ self.weight.T
        if self.bias is not None:
            out = out + self.bias
        return out
    
class BatchNorm2d(nn.Module):
    def __init__(self, dim, eps=1e-5, momentum=0.1, device=None):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.training = True

        self.gamma = nn.Parameter(torch.ones(dim, device=device, requires_grad=True))
        self.beta = nn.Parameter(torch.zeros(dim, device=device, requires_grad=True))

        self.register_buffer("running_mean", torch.zeros(dim))
        self.register_buffer("running_var", torch.ones(dim))

    def forward(self, x):
        if self.training:
            mean = x.mean(dim=(0, 2, 3))
            var = x.var(dim=(0, 2, 3), unbiased=False)
            with torch.no_grad():
                self.running_mean.mul_(1 - self.momentum).add_(self.momentum * mean)
                self.running_var.mul_(1 - self.momentum).add_(self.momentum * var)
        else:
            mean, var = self.running_mean, self.running_var

        shape = (1, -1, 1, 1)
        xhat = (x - mean.view(shape)) / torch.sqrt(var.view(shape) + self.eps)
        return self.gamma.view(shape) * xhat + self.beta.view(shape)


class Tanh(nn.Module):

    def forward(self,x):
        self.out = torch.tanh(x)
        return self.out


class reLU(nn.Module):

    def forward(self, x):
        out = x * (x>0)
        return out

class ResBlock(nn.Module):
    def __init__(self, n_hidden, k):
        super().__init__()
        self.l1 = Linear(n_hidden * k * k, n_hidden)
        self.bn1 = BatchNorm2d(n_hidden)
        self.conv2 = Linear(n_hidden * k * k, n_hidden)
        self.bn2 = BatchNorm2d(n_hidden)
        self.relu = reLU()

    def forward(self, x):
        out = self.bn1(self.l1(x))
        out = self.relu(out)
        out = self.bn2(self.conv2(out))
        return self.relu(out + x)

        


In [94]:
#Net

class ChessNet(nn.Module):

    def __init__(self, n_blocks = 6, n_hidden = 128, n_in = 18, n_final = 73, unfold = 3):
        super().__init__()
        self.linIn = Linear(n_in*unfold * unfold , n_hidden)
        self.bnIn = BatchNorm2d(n_hidden)

        self.blocks = nn.ModuleList([ResBlock(n_hidden, unfold) for i in range(n_blocks)])

        self.policyLinOut = Linear(n_hidden, n_final, unf=False)

        self.valueLinOut = Linear(n_hidden, 1, unf=False)
        self.value_bnOut = BatchNorm2d(1)
        self.value_fc1Out = LinearFlat(64, 256, bias=True)
        self.value_fc2Out = LinearFlat(256, 1, bias=True)
        self.relu = reLU()
        with torch.no_grad():
            for m in model.modules():
                if isinstance(m, Linear) and m is not model.policyLinOut:
                    m.weight *= 2 ** 0.5
            model.policyLinOut.weight *= 0.1

    def forward(self, x):
        B = x.shape[0]
        
        x = self.relu(self.bnIn(self.linIn(x)))

        for block in self.blocks:
            x = block(x)

        logits = self.policyLinOut(x)
        logits = logits.permute(0, 2, 3, 1).reshape(x.shape[0], -1)

        v = self.relu(self.value_bnOut(self.valueLinOut(x)))
        v = v.reshape(B, -1)
        v = self.relu(self.value_fc1Out(v))
        v = torch.tanh(self.value_fc2Out(v))

        return logits, v

    

In [ ]:
model = ChessNet()

parameters = torch.cat([p.detach().flatten() for p in model.parameters()])
print(len(parameters))


1819907


In [82]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

xb = Xtr[:8].float().to(device)
logits, v = model(xb)
print(logits.device, logits.shape)

cuda:0 torch.Size([8, 4672])


In [73]:
import time
for B in (128, 256, 512, 1024):
    idx = torch.randint(0, len(Xtr), (B,))
    xb = Xtr[idx].float().to(device)
    torch.cuda.synchronize(); t = time.time()
    for _ in range(20):
        logits, v = model(xb)
        loss = F.cross_entropy(logits, Ytr[idx].to(device))
        loss.backward()
    torch.cuda.synchronize()
    dt = (time.time() - t) / 20
    print(B, f"{dt*1000:.1f} ms", f"{B/dt:.0f} pos/s")

128 51.2 ms 2502 pos/s
256 77.6 ms 3299 pos/s
512 188.9 ms 2710 pos/s
1024 1088.0 ms 941 pos/s


In [74]:
print(torch.cuda.max_memory_allocated() / 1e9, "Go")

5.553162752 Go


In [ ]:
#training  loop
model = ChessNet().to(device)

max_steps = 300
batch_size = 512
lossi= []
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

for i in range(max_steps):

    #batching

    ix = torch.randint(0, Xtr.shape[0], (batch_size, ))
    Xb, Yb = Xtr[ix].float().to(device), Ytr[ix].to(device)

    #forward pass
    x, v = model(Xb)

    loss = F.cross_entropy(x, Yb)

    #backward pass 
    #for m in model.modules():
    #    if hasattr(m, "out"):
     #       m.out.retain_grad()
    opt.zero_grad(set_to_none=True)
    loss.backward()
   
    #update
    opt.step()


    #tracking stats
    lossi.append(loss.item())
    if i%5000 ==0:
        print(i, loss.item())


    








SyntaxError: expected ':' (2228530148.py, line 9)

In [87]:
model = ChessNet().to(device)

max_steps = 300
batch_size = 512
lossi= []
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

Xs = Xtr[:500].float().to(device)
Ys = Ytr[:500].to(device)

for i in range(400):
    logits, v = model(Xs)
    loss = F.cross_entropy(logits, Ys)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if i % 20 == 0:
        print(i, loss.item())

0 10.919111251831055
20 0.011816609650850296
40 0.0010204619029536843
60 0.0003975427825935185
80 0.00025769072817638516
100 0.00019778158457484096
120 0.00016173133917618543
140 0.0001365688513033092
160 0.00011770269338740036
180 0.00010302412556484342


KeyboardInterrupt: 

In [88]:
model.eval()
with torch.no_grad():
    l, _ = model(Xs)
    print("train en mode eval :", F.cross_entropy(l, Ys).item())
    ld, _ = model(Xdev[:512].float().to(device))
    print("dev :", F.cross_entropy(ld, Ydev[:512].to(device)).item())
model.train()

train en mode eval : 0.00010080346692120656
dev : 8.603192329406738


ChessNet(
  (linIn): Linear()
  (bnIn): BatchNorm2d()
  (blocks): ModuleList(
    (0-5): 6 x ResBlock(
      (l1): Linear()
      (bn1): BatchNorm2d()
      (conv2): Linear()
      (bn2): BatchNorm2d()
      (relu): reLU()
    )
  )
  (policyLinOut): Linear()
  (valueLinOut): Linear()
  (value_bnOut): BatchNorm2d()
  (value_fc1Out): LinearFlat()
  (value_fc2Out): LinearFlat()
  (relu): reLU()
)

In [90]:
print(torch.cuda.memory_allocated()/1e9, torch.cuda.max_memory_allocated()/1e9)

2.011586048 5.553162752


In [96]:
import time

def chrono(f, n=20):
    torch.cuda.synchronize(); t = time.time()
    for _ in range(n): f()
    torch.cuda.synchronize()
    return (time.time() - t) / n * 1000

ix = torch.randint(0, len(Xtr), (512,))
xb, yb = Xtr[ix].float().to(device), Ytr[ix].to(device)

print("données ", chrono(lambda: Xtr[torch.randint(0, len(Xtr), (512,))].float().to(device)))
print("forward  ", chrono(lambda: model(xb)))

def fb():
    logits, v = model(xb)
    loss = F.cross_entropy(logits, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
print("fwd+bwd  ", chrono(fb))

données  0.46509504318237305
forward   136.95571422576904
fwd+bwd   473.8491654396057


In [95]:
lin = model.linIn
xt = Xtr[:8].float().to(device)

C = lin.weight.shape[1] // 9
a = (lin.weight @ unfoldX(xt)).view(8, -1, 8, 8)
b = F.conv2d(xt, lin.weight.view(-1, C, 3, 3), padding=1)
print((a - b).abs().max().item())

0.0003701746463775635


4.805264472961426


ChessNet(
  (linIn): Linear()
  (bnIn): BatchNorm2d()
  (blocks): ModuleList(
    (0-5): 6 x ResBlock(
      (l1): Linear()
      (bn1): BatchNorm2d()
      (conv2): Linear()
      (bn2): BatchNorm2d()
      (relu): reLU()
    )
  )
  (policyLinOut): Linear()
  (valueLinOut): Linear()
  (value_bnOut): BatchNorm2d()
  (value_fc1Out): LinearFlat()
  (value_fc2Out): LinearFlat()
  (relu): reLU()
)

In [102]:
ckpt = torch.load("../runs/ckpt.pt", map_location=device)
print(ckpt["step"])

14000


In [104]:
ckpt = torch.load("../runs/ckpt.pt", map_location=device)
model.load_state_dict(ckpt["model"])
opt.load_state_dict(ckpt["opt"])
model.train()

ChessNet(
  (linIn): Linear()
  (bnIn): BatchNorm2d()
  (blocks): ModuleList(
    (0-5): 6 x ResBlock(
      (l1): Linear()
      (bn1): BatchNorm2d()
      (conv2): Linear()
      (bn2): BatchNorm2d()
      (relu): reLU()
    )
  )
  (policyLinOut): Linear()
  (valueLinOut): Linear()
  (value_bnOut): BatchNorm2d()
  (value_fc1Out): LinearFlat()
  (value_fc2Out): LinearFlat()
  (relu): reLU()
)

In [105]:
model.eval()
with torch.no_grad():
    jx = torch.randint(0, Xdev.shape[0], (2048,))
    ld, _ = model(Xdev[jx].float().to(device))
    print(F.cross_entropy(ld, Ydev[jx].to(device)).item())
model.train()

2.0861124992370605


ChessNet(
  (linIn): Linear()
  (bnIn): BatchNorm2d()
  (blocks): ModuleList(
    (0-5): 6 x ResBlock(
      (l1): Linear()
      (bn1): BatchNorm2d()
      (conv2): Linear()
      (bn2): BatchNorm2d()
      (relu): reLU()
    )
  )
  (policyLinOut): Linear()
  (valueLinOut): Linear()
  (value_bnOut): BatchNorm2d()
  (value_fc1Out): LinearFlat()
  (value_fc2Out): LinearFlat()
  (relu): reLU()
)